In [2]:
!pip install sqlite-vec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 6.5 MB/s eta 0:00:00


In [3]:
import sqlite3
import sqlite_vec
import hashlib
import json
import numpy as np
from typing import List

This cell installs the `sqlite-vec` library, which is an SQLite extension for vector search.

```python
!pip install sqlite-vec
```
- `!pip install sqlite-vec`: This command is executed in the shell (due to the `!` prefix) and downloads and installs the `sqlite-vec` Python package, which includes the SQLite extension.

This cell imports necessary libraries for database interaction, vector operations, and data handling.

```python
import sqlite3
import sqlite_vec
import hashlib
import json
import numpy as np
from typing import List
```
- `import sqlite3`: Imports the standard Python library for interacting with SQLite databases.
- `import sqlite_vec`: Imports the `sqlite-vec` Python wrapper, which provides functionality to load the SQLite vector extension.
- `import hashlib`: Imports the `hashlib` module, which will be used to generate cryptographic hashes (specifically SHA256) for creating deterministic embeddings.
- `import json`: Imports the `json` module, which is used to serialize Python objects (like lists of floats) into JSON strings, required for storing and querying vectors in `sqlite-vec`.
- `import numpy as np`: Imports the NumPy library, aliased as `np`, which is crucial for numerical operations, especially for handling arrays of floating-point numbers that represent vectors.
- `from typing import List`: Imports the `List` type hint from the `typing` module, used for better code readability and type checking, indicating that a function will return a list.

This cell establishes a connection to an SQLite database and loads the `sqlite-vec` extension.

```python
# Step 1 - Connection SQLite
conn = sqlite3.connect("demo.db")
conn.enable_load_extension(True)
sqlite_vec.load(conn)
cur = conn.cursor()
```
- `# Step 1 - Connection SQLite`: A comment indicating the purpose of the following code.
- `conn = sqlite3.connect("demo.db")`: This line establishes a connection to an SQLite database file named `demo.db`. If the file doesn't exist, it will be created. The connection object is stored in the `conn` variable.
- `conn.enable_load_extension(True)`: This line enables the ability to load SQLite extensions for the current connection. This is a crucial step for `sqlite-vec` to function.
- `sqlite_vec.load(conn)`: This line explicitly loads the `sqlite-vec` extension into the SQLite database connection represented by `conn`.
- `cur = conn.cursor()`: This line creates a cursor object from the connection. A cursor is used to execute SQL commands and fetch results from the database.

This cell creates a table to store documents and populates it with sample data if it's currently empty.

```python
# Step 2 - Create Tables
cur.execute("""
CREATE TABLE IF NOT EXISTS docs (
    id INTEGER PRIMARY KEY,
    content TEXT NOT NULL
);
""")

# Insert docs only if table is empty (idempotent)
cur.execute("SELECT COUNT(*) FROM docs;")
docs = [
    (1, "The quick brown fox jumps over the lazy dog"),
    (2, "A fast auburn fox leaps above a sleepy canine"),
    (3, "An article about database systems and vector search"),
    (4, "Deep learning and embeddings for natural language processing"),
]
docs_count = cur.fetchone()[0]
if docs_count == 0:
    cur.executemany("INSERT INTO docs(id, content) VALUES (?, ?);", docs)
```
- `# Step 2 - Create Tables`: A comment indicating the purpose of the following code.
- `cur.execute("""...""")`: This executes an SQL command to create a table named `docs`.
    - `CREATE TABLE IF NOT EXISTS docs`: Ensures the table is only created if it doesn't already exist, preventing errors on repeated execution.
    - `id INTEGER PRIMARY KEY`: Defines an `id` column as an integer and sets it as the primary key, ensuring uniqueness and providing an efficient way to identify each document.
    - `content TEXT NOT NULL`: Defines a `content` column to store the document text, ensuring it cannot be empty.
- `# Insert docs only if table is empty (idempotent)`: A comment explaining the idempotent nature of the next block.
- `cur.execute("SELECT COUNT(*) FROM docs;")`: Executes a query to count the number of rows in the `docs` table.
- `docs = [...]`: Defines a list of tuples, where each tuple represents a document with an `id` and `content`.
- `docs_count = cur.fetchone()[0]`: Fetches the result of the `COUNT(*)` query (which is a single row with one column) and extracts the count, storing it in `docs_count`.
- `if docs_count == 0:`: This condition checks if the `docs` table is empty.
- `cur.executemany("INSERT INTO docs(id, content) VALUES (?, ?);", docs)`: If the table is empty, this line inserts all the documents from the `docs` list into the `docs` table. `executemany` is efficient for inserting multiple rows.

This cell defines a function `embed_text` that generates a simple, deterministic vector embedding for a given text.

```python
def embed_text(text: str, dim: int = 8) -> List[float]:
  """Simple deterministic embedding for demo purposes.
  Uses SHA256 of the text to produce a repeatable vector in range [-1, 1].
  """
  h = hashlib.sha256(text.encode("utf-8")).digest()
  vec = []
  for i in range(dim):
      b1 = h[(i * 2) % len(h)]
      b2 = h[(i * 2 + 1) % len(h)]
      val = (b1 << 8) | b2
      f = (val / 65535.0) * 2.0 - 1.0
      vec.append(f)
  return vec
```
- `def embed_text(text: str, dim: int = 8) -> List[float]:`: Defines a function named `embed_text` that takes two arguments:
    - `text`: A string representing the input text to be embedded.
    - `dim`: An integer representing the desired dimension of the output vector, defaulting to 8.
    - `-> List[float]`: A type hint indicating that the function will return a list of floating-point numbers.
- `"""Simple deterministic embedding..."""`: A docstring explaining the function's purpose.
- `h = hashlib.sha256(text.encode("utf-8")).digest()`: This line calculates a SHA256 hash of the input `text`.
    - `text.encode("utf-8")`: Converts the input string into a sequence of bytes using UTF-8 encoding, as hash functions operate on bytes.
    - `hashlib.sha256(...)`: Creates a SHA256 hash object.
    - `.digest()`: Returns the hash as a bytes object (a sequence of bytes).
- `vec = []`: Initializes an empty list `vec` which will store the generated vector components.
- `for i in range(dim):`: This loop iterates `dim` times, creating each component of the vector.
    - `b1 = h[(i * 2) % len(h)]`: Extracts a byte from the hash `h`. The modulo operator `% len(h)` ensures that the index wraps around if `dim` is larger than half the hash length, making the process deterministic and using all hash bytes.
    - `b2 = h[(i * 2 + 1) % len(h)]`: Extracts the next byte from the hash, similar to `b1`.
    - `val = (b1 << 8) | b2`: Combines the two bytes `b1` and `b2` into a 16-bit integer.
        - `b1 << 8`: Shifts `b1` 8 bits to the left, effectively multiplying it by 256 and placing it in the higher byte position.
        - `| b2`: Performs a bitwise OR operation with `b2`, placing `b2` in the lower byte position. This forms a 16-bit unsigned integer (0-65535).
    - `f = (val / 65535.0) * 2.0 - 1.0`: Normalizes the `val` (which is between 0 and 65535) into a floating-point number between -1.0 and 1.0.
        - `val / 65535.0`: Scales `val` to a range of 0.0 to 1.0.
        - `* 2.0`: Scales it to 0.0 to 2.0.
        - `- 1.0`: Shifts it to -1.0 to 1.0.
    - `vec.append(f)`: Appends the calculated float `f` to the `vec` list.
- `return vec`: Returns the completed list of floats, which is the generated vector embedding.

This cell creates a virtual table for vector embeddings using `sqlite-vec` and populates it with embeddings generated from the `docs` table.

```python
# Step 3 - Add vector embeddings
cur.execute("""
CREATE VIRTUAL TABLE IF NOT EXISTS vec_docs USING vec0(
    embedding FLOAT[8]
);
""")
cur.execute("SELECT COUNT(*) FROM vec_docs;")
vec_count = cur.fetchone()[0]
if vec_count == 0:
  rows = []
  for _id, text in docs:
      emb = embed_text(text, dim=8)
      rows.append((_id, json.dumps(emb)))

  cur.executemany("INSERT INTO vec_docs(rowid, embedding) VALUES (?, ?);", rows)
  conn.commit()
```
- `# Step 3 - Add vector embeddings`: A comment indicating the purpose of the following code.
- `cur.execute("""...""")`: Executes an SQL command to create a virtual table.
    - `CREATE VIRTUAL TABLE IF NOT EXISTS vec_docs USING vec0(...)`: This is the core `sqlite-vec` command. It creates a virtual table named `vec_docs` using the `vec0` module.
    - `embedding FLOAT[8]`: Defines a column named `embedding` which will store 8-dimensional floating-point vectors. `vec0` automatically handles the storage and indexing for this type.
- `cur.execute("SELECT COUNT(*) FROM vec_docs;")`: Counts the number of rows in the `vec_docs` virtual table.
- `vec_count = cur.fetchone()[0]`: Retrieves the count from the query result.
- `if vec_count == 0:`: This condition ensures that embeddings are only generated and inserted if the `vec_docs` table is currently empty, making the operation idempotent.
- `rows = []`: Initializes an empty list to store the `(rowid, embedding)` pairs.
- `for _id, text in docs:`: Iterates through each document in the previously defined `docs` list (which contains `(id, content)` tuples).
    - `emb = embed_text(text, dim=8)`: Calls the `embed_text` function to generate an 8-dimensional vector embedding for the current document's `text`.
    - `rows.append((_id, json.dumps(emb)))`: Appends a tuple to the `rows` list. The tuple contains:
        - `_id`: The original `id` of the document.
        - `json.dumps(emb)`: The generated embedding `emb` is converted into a JSON string. `sqlite-vec` requires vector inputs as JSON strings for insertion.
- `cur.executemany("INSERT INTO vec_docs(rowid, embedding) VALUES (?, ?);", rows)`: Inserts all the prepared `(rowid, embedding)` pairs into the `vec_docs` virtual table. The `rowid` of the `vec_docs` virtual table implicitly links back to the `id` from the `docs` table.
- `conn.commit()`: Commits the current transaction to the database, permanently saving the inserted vector embeddings.

This cell demonstrates how to retrieve vector embeddings from the `vec_docs` table and convert them back into NumPy arrays.

```python
cur = conn.execute("SELECT rowid, embedding FROM vec_docs LIMIT 5;")
for row in cur.fetchall():
    rowid, blob = row
    vec = np.frombuffer(blob, dtype=np.float32)
    print(f"id={rowid}, vec={vec[:8]}") # Print first 8 dimensions
```
- `cur = conn.execute("SELECT rowid, embedding FROM vec_docs LIMIT 5;")`: Executes an SQL query to select the `rowid` and the `embedding` (as a binary BLOB) from the `vec_docs` virtual table. `LIMIT 5` restricts the output to the first five entries.
- `for row in cur.fetchall():`: Iterates through each row returned by the query. `cur.fetchall()` retrieves all remaining rows of a query result set and returns them as a list of tuples.
- `rowid, blob = row`: Unpacks the current `row` tuple into two variables: `rowid` (the integer ID) and `blob` (the binary representation of the vector embedding).
- `vec = np.frombuffer(blob, dtype=np.float32)`: Converts the binary `blob` data back into a NumPy array.
    - `np.frombuffer()`: Creates a new one-dimensional array from a buffer (like the `blob` bytes).
    - `dtype=np.float32`: Specifies that the data within the buffer should be interpreted as 32-bit floating-point numbers.
- `print(f"id={rowid}, vec={vec[:8]}")`: Prints the document's `rowid` and the first 8 elements of the reconstructed NumPy vector `vec`. (Since our vectors are 8-dimensional, `vec[:8]` shows the whole vector).

This cell demonstrates how to perform a vector similarity search using `sqlite-vec` with a query string.

```python
# Step 4 - Test query
query = "fox dog"
text_query = query
query_vec = embed_text(text_query, dim=8)
query_vec_json = json.dumps(query_vec)
res = cur.execute("""
    SELECT rowid, distance
    FROM vec_docs
    WHERE embedding MATCH ?

    and k=2;
    """,
    (query_vec_json,)).fetchall()

for rowid, distance in res:
    print(f"- rowid={rowid}  distance={float(distance):.12f}")
```
- `# Step 4 - Test query`: A comment indicating the purpose of the following code.
- `query = "fox dog"`: Defines the natural language query string that we want to embed and search for.
- `text_query = query`: Assigns the `query` string to `text_query` (redundant here, but good practice if `query` were to be modified later).
- `query_vec = embed_text(text_query, dim=8)`: Generates an 8-dimensional vector embedding for the `text_query` using our `embed_text` function.
- `query_vec_json = json.dumps(query_vec)`: Converts the generated `query_vec` (a list of floats) into a JSON string. This format is required by `sqlite-vec` when passing query vectors to the `MATCH` operator.
- `res = cur.execute("""...""", (query_vec_json,)).fetchall()`: Executes the SQL query to perform the vector similarity search.
    - `SELECT rowid, distance`: Selects the `rowid` of the matching documents and their calculated `distance` from the query vector.
    - `FROM vec_docs`: Specifies that the search is performed on the `vec_docs` virtual table.
    - `WHERE embedding MATCH ?`: This is the core `sqlite-vec` syntax for performing a vector search. The `?` is a placeholder for the query vector.
    - `and k=2`: An argument passed to the `MATCH` operator, indicating that we want to retrieve only the top `k=2` closest results.
    - `(query_vec_json,)`: This is a tuple containing the JSON string of our query vector, which replaces the `?` placeholder in the SQL query.
    - `.fetchall()`: Retrieves all matching rows from the query result.
- `for rowid, distance in res:`: Iterates through each `(rowid, distance)` tuple returned by the search.
- `print(f"- rowid={rowid}  distance={float(distance):.12f}")`: Prints the `rowid` of the retrieved document and its distance to the query vector, formatted to 12 decimal places for precision.

In [4]:
# Step 1 - Connection SQLite
conn = sqlite3.connect("demo.db")
conn.enable_load_extension(True)
sqlite_vec.load(conn)
cur = conn.cursor()

In [5]:
# Step 2 - Create Tables
cur.execute("""
CREATE TABLE IF NOT EXISTS docs (
    id INTEGER PRIMARY KEY,
    content TEXT NOT NULL
);
""")

# Insert docs only if table is empty (idempotent)
cur.execute("SELECT COUNT(*) FROM docs;")
docs = [
    (1, "The quick brown fox jumps over the lazy dog"),
    (2, "A fast auburn fox leaps above a sleepy canine"),
    (3, "An article about database systems and vector search"),
    (4, "Deep learning and embeddings for natural language processing"),
]
docs_count = cur.fetchone()[0]
if docs_count == 0:
    cur.executemany("INSERT INTO docs(id, content) VALUES (?, ?);", docs)

In [6]:
def embed_text(text: str, dim: int = 8) -> List[float]:
  """Simple deterministic embedding for demo purposes.
  Uses SHA256 of the text to produce a repeatable vector in range [-1, 1].
  """
  h = hashlib.sha256(text.encode("utf-8")).digest()
  vec = []
  for i in range(dim):
      b1 = h[(i * 2) % len(h)]
      b2 = h[(i * 2 + 1) % len(h)]
      val = (b1 << 8) | b2
      f = (val / 65535.0) * 2.0 - 1.0
      vec.append(f)
  return vec

In [7]:
# Step 3 - Add vector embeddings
cur.execute("""
CREATE VIRTUAL TABLE IF NOT EXISTS vec_docs USING vec0(
    embedding FLOAT[8]
);
""")
cur.execute("SELECT COUNT(*) FROM vec_docs;")
vec_count = cur.fetchone()[0]
if vec_count == 0:
  rows = []
  for _id, text in docs:
      emb = embed_text(text, dim=8)
      rows.append((_id, json.dumps(emb)))

  cur.executemany("INSERT INTO vec_docs(rowid, embedding) VALUES (?, ?);", rows)
  conn.commit()

In [8]:
cur = conn.execute("SELECT rowid, embedding FROM vec_docs LIMIT 5;")
for row in cur.fetchall():
    rowid, blob = row
    vec = np.frombuffer(blob, dtype=np.float32)
    print(f"id={rowid}, vec={vec[:8]}") # Print first 8 dimensions

id=1, vec=[ 0.68484014  0.9664301  -0.93875027  0.00453193 -0.17351034  0.20888075
  0.37526512 -0.63820857]
id=2, vec=[-0.10630961  0.9881895   0.983032   -0.3600061  -0.8277867   0.31184864
 -0.8834821  -0.96121156]
id=3, vec=[-0.5786679  -0.48274967 -0.9851072   0.9522393   0.95977724  0.27382314
  0.81451136 -0.19572747]
id=4, vec=[-0.6383917   0.20823987 -0.6133974  -0.80352485  0.38023958 -0.9358206
 -0.67974365  0.78530556]


In [12]:
# Step 4 - Test query
query = "fox dog"
text_query = query
query_vec = embed_text(text_query, dim=8)
query_vec_json = json.dumps(query_vec)
res = cur.execute("""
    SELECT rowid, distance
    FROM vec_docs
    WHERE embedding MATCH ?

    and k=2;
    """,
    (query_vec_json,)).fetchall()

for rowid, distance in res:
    print(f"- rowid={rowid}  distance={float(distance):.12f}")

- rowid=3  distance=1.915137887001
- rowid=1  distance=2.211798191071
